# Opgave 1

## Spm. 1)

In [7]:
from modeller.transport_problem import TransportProblem
import numpy as np
import pulp as PLP

demands = [225, 125, 150, 325, 175]
capacities = [400, 200, 300 , 100]
cost_matrix = np.array([[3,6,6,7,7],
                        [2,9,4,9,4],
                        [8,8,3,1,7],
                        [5,4,8,5,2]])
tp = TransportProblem(cost_matrix, capacities, demands)
tp.print_transport_details()

SUPPLY MEETS DEMAND

--- Transport Route Details ---
Supplier 0 sends 225.0 units to Customer 0 | Unit Cost: 3 | Route Cost: 675.0
Supplier 0 sends 125.0 units to Customer 1 | Unit Cost: 6 | Route Cost: 750.0
Supplier 0 sends 25.0 units to Customer 2 | Unit Cost: 6 | Route Cost: 150.0
Supplier 0 sends 25.0 units to Customer 3 | Unit Cost: 7 | Route Cost: 175.0
Supplier 0 sends 0.0 units to Customer 4 | Unit Cost: 7 | Route Cost: 0.0
Supplier 1 sends 0.0 units to Customer 0 | Unit Cost: 2 | Route Cost: 0.0
Supplier 1 sends 0.0 units to Customer 1 | Unit Cost: 9 | Route Cost: 0.0
Supplier 1 sends 125.0 units to Customer 2 | Unit Cost: 4 | Route Cost: 500.0
Supplier 1 sends 0.0 units to Customer 3 | Unit Cost: 9 | Route Cost: 0.0
Supplier 1 sends 75.0 units to Customer 4 | Unit Cost: 4 | Route Cost: 300.0
Supplier 2 sends 0.0 units to Customer 0 | Unit Cost: 8 | Route Cost: 0.0
Supplier 2 sends 0.0 units to Customer 1 | Unit Cost: 8 | Route Cost: 0.0
Supplier 2 sends 0.0 units to Customer

## Spm 4.)
Jeg følger slides fra uge 11 Kap9 - Heltalsmodeller
Vi kan løse dette problem ved at indføre en indikator variabel $\delta$ der indikerer om der sendes noget fra den kontinuerte $x_{ij}$
variabel. Dette gøres jf slide 6/22 at lave en begrænsning således at:
$$x \leq M\delta \qquad x \geq m\delta$$
Hvor M er en øvre grænse for et vilkårligt x og m er en positiv nedre grænse.


In [8]:
tpILP = TransportProblem(cost_matrix, capacities, demands)

# Extend model with binary indicator variables delta
tpILP.delta = PLP.LpVariable.dicts("delta", indices= (tpILP.supplier_range, tpILP.demands_range), cat = PLP.LpBinary)
m = 10**-3
M = 10**3

# Make indicator variable 1 if there is flow from supplier i to costumer j, otherwise 0
for i in tpILP.supplier_range:
    for j in tpILP.demands_range:
        tpILP.model += tpILP.x[i][j] <= M*tpILP.delta[i][j]
        tpILP.model += tpILP.x[i][j] >= m*tpILP.delta[i][j]
# Add restriction that for supplier 1 (0) we must only send to maximum two costumers
i = 0
tpILP.model += PLP.lpSum(tpILP.delta[i][j] for j in tpILP.demands_range) == 2
tpILP.print_transport_details(one_indexed=True)


#

SUPPLY MEETS DEMAND

--- Transport Route Details ---

--- Transport Route Details (1-indexed) ---
Supplier 1 sends 225.0 units to Customer 1 | Unit Cost: 3 | Route Cost: 675.0
Supplier 1 sends 0.0 units to Customer 2 | Unit Cost: 6 | Route Cost: 0.0
Supplier 1 sends 0.0 units to Customer 3 | Unit Cost: 6 | Route Cost: 0.0
Supplier 1 sends 0.0 units to Customer 4 | Unit Cost: 7 | Route Cost: 0.0
Supplier 1 sends 175.0 units to Customer 5 | Unit Cost: 7 | Route Cost: 1225.0
Supplier 2 sends 0.0 units to Customer 1 | Unit Cost: 2 | Route Cost: 0.0
Supplier 2 sends 25.0 units to Customer 2 | Unit Cost: 9 | Route Cost: 225.0
Supplier 2 sends 150.0 units to Customer 3 | Unit Cost: 4 | Route Cost: 600.0
Supplier 2 sends 25.0 units to Customer 4 | Unit Cost: 9 | Route Cost: 225.0
Supplier 2 sends 0.0 units to Customer 5 | Unit Cost: 4 | Route Cost: 0.0
Supplier 3 sends 0.0 units to Customer 1 | Unit Cost: 8 | Route Cost: 0.0
Supplier 3 sends 0.0 units to Customer 2 | Unit Cost: 8 | Route Cost:

## Spm 5.)
Vi skal her introducere en fixed charge, jf. uge 11 Kap9 - Heltalsmodeller slide 7, kan vi opnå dette ved at modificere objektfunktionen i problemet med en fixed charge $\gamma F$, samt ved brug af en indikatorvariabel $\gamma$ underlagt begrænsningen $\gamma = 1  \iff \sum(\delta) > 2, \, 0 \text{ ellers}$.

In [9]:
tpFixed = TransportProblem(cost_matrix, capacities, demands)

#Update the model objective
tpFixed.gamma = PLP.LpVariable("gamma", cat = PLP.LpBinary)
F = 100
tpFixed.model.objective  = tpFixed.model.objective + F * tpFixed.gamma

# Extend model with binary indicator variables delta
tpFixed.delta = PLP.LpVariable.dicts("delta", indices= (tpFixed.supplier_range, tpFixed.demands_range), cat = PLP.LpBinary)
m = 10**-3
M = 10**3

# Make indicator variable 1 if there is flow from supplier i to costumer j, otherwise 0
for i in tpFixed.supplier_range:
    for j in tpFixed.demands_range:
        tpFixed.model += tpFixed.x[i][j] <= M*tpFixed.delta[i][j]
        tpFixed.model += tpFixed.x[i][j] >= m*tpFixed.delta[i][j]

i = 0
# We have to inforce if sum of deltas > 2, then gamma = 1, since we minimize and gamma adds a fixed cost
# The model will just set gamma = 0 if possible, as this eliminates cost, but if sum of deltas is >= 3, then gamme must be 1.
tpFixed.model += PLP.lpSum(tpFixed.delta[i][j] for j in tpFixed.demands_range) <= 2 + tpFixed.gamma * M

tpFixed.print_transport_details(one_indexed=True)
# small value to account for numerical error
eps = 10**-3
print("Fixed cost used:",tpFixed.gamma.varValue > 0 + eps)

SUPPLY MEETS DEMAND

--- Transport Route Details ---

--- Transport Route Details (1-indexed) ---
Supplier 1 sends 225.0 units to Customer 1 | Unit Cost: 3 | Route Cost: 675.0
Supplier 1 sends 125.0 units to Customer 2 | Unit Cost: 6 | Route Cost: 750.0
Supplier 1 sends 25.0 units to Customer 3 | Unit Cost: 6 | Route Cost: 150.0
Supplier 1 sends 25.0 units to Customer 4 | Unit Cost: 7 | Route Cost: 175.0
Supplier 1 sends 0.0 units to Customer 5 | Unit Cost: 7 | Route Cost: 0.0
Supplier 2 sends 0.0 units to Customer 1 | Unit Cost: 2 | Route Cost: 0.0
Supplier 2 sends 0.0 units to Customer 2 | Unit Cost: 9 | Route Cost: 0.0
Supplier 2 sends 125.0 units to Customer 3 | Unit Cost: 4 | Route Cost: 500.0
Supplier 2 sends 0.0 units to Customer 4 | Unit Cost: 9 | Route Cost: 0.0
Supplier 2 sends 75.0 units to Customer 5 | Unit Cost: 4 | Route Cost: 300.0
Supplier 3 sends 0.0 units to Customer 1 | Unit Cost: 8 | Route Cost: 0.0
Supplier 3 sends 0.0 units to Customer 2 | Unit Cost: 8 | Route Cos

## Spm. 6)
Dette problem kan løses ved at introducere stordriftsfordele (Slides uge 11 Kap9 - Heltalsmodeller)

Vi kan udelede prisen på følgende måde. Prisstigningen kan beskrives som ændringen i prisen delt med ændringen af den brugte mængde, altså:
$$\Delta Pris_j = \dfrac{c_j-c_{j-1}}{b_j-b_{j-1}}$$
Vi kan så bruge denne prisstigning i det j'te interval til at beskrive den totale pris for at sende en fraktion $f_j$ af efterspørgslen på linje segment j, som:
$$C(f) = \sum_{j=1}^n \Delta Pris_j  (b_j - b_{j-1})f_j =   \sum_{j=1}^n (c_j - c_{j-1})f_j $$

Hvor $f_j$ er den fraktion af efterspørgslen der sendes på linje segment j, og $c_j$ er prisen for at sende på linje segment j. Vi har her indført en dummy linje segment 0, hvor $c_0 = 0$ og $f_0 = 0$.

In [10]:
tpLarge = TransportProblem(cost_matrix, capacities, demands)
# Large number
M = 10**6
# Number of line segements
n = 3
# delta and f range
df_range = range(1, n + 1)
# price intervals
b = [0, 100, 50, M]
# costs
costs = [0, 4, 3, 2]
# fracton of demand that is sent on each line segment is introduced as a variable
f = PLP.LpVariable.dict("f", df_range, cat = PLP.LpContinuous, lowBound=0, upBound= 1)
x_large = PLP.lpSum(f[j] * b[j] - b[j-1]  for j in df_range)

# Introduce the pricing to the objective
C = PLP.lpSum((costs[j] - costs[j - 1])*f[j] for j in df_range)

tpLarge.model.objective = tpLarge.model.objective + C

# We introduce delta variables to indicate if we are sending on line segment j
delta = PLP.LpVariable.dict("delta", range(1,n + 1), cat = PLP.LpBinary)
for j in df_range:
    # Indicates that the fraction is used
    tpLarge.model += f[j] <= delta[j]
for j in df_range[1:]:
    # Indicates that previous fraction must be used before we can use the next
    tpLarge.model += f[j-1] >= delta[j]

tpLarge.print_transport_details(one_indexed=True)

SUPPLY MEETS DEMAND

--- Transport Route Details ---

--- Transport Route Details (1-indexed) ---
Supplier 1 sends 225.0 units to Customer 1 | Unit Cost: 3 | Route Cost: 675.0
Supplier 1 sends 125.0 units to Customer 2 | Unit Cost: 6 | Route Cost: 750.0
Supplier 1 sends 25.0 units to Customer 3 | Unit Cost: 6 | Route Cost: 150.0
Supplier 1 sends 25.0 units to Customer 4 | Unit Cost: 7 | Route Cost: 175.0
Supplier 1 sends 0.0 units to Customer 5 | Unit Cost: 7 | Route Cost: 0.0
Supplier 2 sends 0.0 units to Customer 1 | Unit Cost: 2 | Route Cost: 0.0
Supplier 2 sends 0.0 units to Customer 2 | Unit Cost: 9 | Route Cost: 0.0
Supplier 2 sends 125.0 units to Customer 3 | Unit Cost: 4 | Route Cost: 500.0
Supplier 2 sends 0.0 units to Customer 4 | Unit Cost: 9 | Route Cost: 0.0
Supplier 2 sends 75.0 units to Customer 5 | Unit Cost: 4 | Route Cost: 300.0
Supplier 3 sends 0.0 units to Customer 1 | Unit Cost: 8 | Route Cost: 0.0
Supplier 3 sends 0.0 units to Customer 2 | Unit Cost: 8 | Route Cos

## Spm. 7)
Vi kan løse denne problemstilling som transportproblem ved a indføre en dummy leverandør og en dummy kunde, hvor det at gratis at sende fra den dummy leverandør til dummy kunde, men de men vi introducerer de respektive priser og til de andre kunder og leverandører. Vi sørger for at balancerer problemet ved at sætte dummy leverandørens kapacitet og dummy til den angive kapacitet.



tpTranshipment = TransportProblem(cost_matrix, capcities, demands)

In [11]:
# Given values
T_cap = 250
prices_from_supplier_to_T = np.array([2, 3, 1, 1])
prices_from_T_to_costumer = np.array([1, 4, 3, 1, 2])
# Extend demands and capacities
demands = [225, 125, 150, 325, 175, T_cap]
capacities = [400, 200, 300 , 100, T_cap]

cost_matrix = np.array([[3,6,6,7,7],
                        [2,9,4,9,4],
                        [8,8,3,1,7],
                        [5,4,8,5,2]])
# Extend cost matrix with dummy supplier and dummy costumer
temp = np.zeros((cost_matrix.shape[0] + 1, cost_matrix.shape[1] + 1))
temp[:-1, :-1] = cost_matrix
# Add prices from suppliers to dummy costumer as last column
temp[:-1, -1] = prices_from_supplier_to_T
# Add prices from dummy supplier to costumers as last row
temp[-1, :-1] = prices_from_T_to_costumer
cost_matrix = temp
print(cost_matrix)

[[3. 6. 6. 7. 7. 2.]
 [2. 9. 4. 9. 4. 3.]
 [8. 8. 3. 1. 7. 1.]
 [5. 4. 8. 5. 2. 1.]
 [1. 4. 3. 1. 2. 0.]]


Vi kan nu løse dette transshipment problem som et transportproblem, hvor den sidste leverandør og den sidste kunde er vores dummy leverandør og dummy kunde. Vi sørger for at balancere problemet ved at sætte dummy leverandørens kapacitet og dummy til den angive kapacitet.

In [19]:
tpTranshipment = TransportProblem(cost_matrix, capacities, demands)
tpTranshipment.print_transport_details(one_indexed=True)

SUPPLY MEETS DEMAND

--- Transport Route Details ---

--- Transport Route Details (1-indexed) ---
Supplier 1 sends 175.0 units to Customer 1 | Unit Cost: 3.0 | Route Cost: 525.0
Supplier 1 sends 125.0 units to Customer 2 | Unit Cost: 6.0 | Route Cost: 750.0
Supplier 1 sends 0.0 units to Customer 3 | Unit Cost: 6.0 | Route Cost: 0.0
Supplier 1 sends 0.0 units to Customer 4 | Unit Cost: 7.0 | Route Cost: 0.0
Supplier 1 sends 0.0 units to Customer 5 | Unit Cost: 7.0 | Route Cost: 0.0
Supplier 1 sends 100.0 units to Customer 6 | Unit Cost: 2.0 | Route Cost: 200.0
Supplier 2 sends 50.0 units to Customer 1 | Unit Cost: 2.0 | Route Cost: 100.0
Supplier 2 sends 0.0 units to Customer 2 | Unit Cost: 9.0 | Route Cost: 0.0
Supplier 2 sends 150.0 units to Customer 3 | Unit Cost: 4.0 | Route Cost: 600.0
Supplier 2 sends 0.0 units to Customer 4 | Unit Cost: 9.0 | Route Cost: 0.0
Supplier 2 sends 0.0 units to Customer 5 | Unit Cost: 4.0 | Route Cost: 0.0
Supplier 2 sends 0.0 units to Customer 6 | Unit